In [7]:
from PIL import Image, ImageDraw
import numpy as np
from collections import deque
import colorsys
from typing import List
from enum import Enum
import math

# ================================================================
# Constants
# ================================================================
BORDER = 2
CELL_SIZE = 14
STRIDE = 16
NUM_CELLS = 64
MAT_SIZE = 128

# Matrix codes
EMPTY, WALL, START, GOAL, DEATH_PIT, TELEPORT, CONFUSION = 0, 1, 2, 3, 4, 5, 6

CELL_NAMES = {
    EMPTY: "Empty",
    WALL: "Wall",
    START: "Start",
    GOAL: "Goal",
    DEATH_PIT: "Death Pit",
    TELEPORT: "Teleport",
    CONFUSION: "Confusion",
}

CELL_COLORS = {
    EMPTY: (255, 255, 255),
    WALL: (0, 0, 0),
    START: (0, 200, 0),
    GOAL: (30, 100, 220),
    DEATH_PIT: (220, 50, 0),
    TELEPORT: (0, 190, 190),
    CONFUSION: (160, 0, 200),
}


# ================================================================
# Basic classes
# ================================================================
class Action(Enum):
    MOVE_UP = 0
    MOVE_DOWN = 1
    MOVE_LEFT = 2
    MOVE_RIGHT = 3
    WAIT = 4


class TurnResult:
    def __init__(self):
        self.wall_hits = 0
        self.current_position = (0, 0)
        self.is_dead = False
        self.is_confused = False
        self.is_goal_reached = False
        self.teleported = False
        self.actions_executed = 0

    def __repr__(self):
        return (
            f"pos={self.current_position} dead={self.is_dead} "
            f"goal={self.is_goal_reached} confused={self.is_confused} "
            f"teleported={self.teleported} wall_hits={self.wall_hits} "
            f"actions_executed={self.actions_executed}"
        )


# ================================================================
# 1. Convert MAZE_0 image into matrix data
# ================================================================
def _wall_below(gray, row, col):
    y = BORDER + row * STRIDE + CELL_SIZE
    x = BORDER + col * STRIDE + CELL_SIZE // 2
    return not (gray[y, x] > 128 and gray[y + 1, x] > 128)


def _wall_right(gray, row, col):
    y = BORDER + row * STRIDE + CELL_SIZE // 2
    x = BORDER + col * STRIDE + CELL_SIZE
    return not (gray[y, x] > 128 and gray[y, x + 1] > 128)


def load_maze(image_path):
    gray = np.array(Image.open(image_path).convert("L"))
    matrix = np.ones((MAT_SIZE, MAT_SIZE), dtype=np.uint8)

    for r in range(NUM_CELLS):
        for c in range(NUM_CELLS):
            matrix[r * 2, c * 2] = EMPTY

            if r < NUM_CELLS - 1:
                matrix[r * 2 + 1, c * 2] = WALL if _wall_below(gray, r, c) else EMPTY

            if c < NUM_CELLS - 1:
                matrix[r * 2, c * 2 + 1] = WALL if _wall_right(gray, r, c) else EMPTY

            if r < NUM_CELLS - 1 and c < NUM_CELLS - 1:
                matrix[r * 2 + 1, c * 2 + 1] = WALL

    return matrix


# ================================================================
# 2. Detect hazards from MAZE_1 image
# ================================================================
def detect_hazards(image_path):
    rgb = np.array(Image.open(image_path).convert("RGB"))
    hazards = {}

    for r in range(NUM_CELLS):
        for c in range(NUM_CELLS):
            cy = BORDER + r * STRIDE + CELL_SIZE // 2
            cx = BORDER + c * STRIDE + CELL_SIZE // 2

            colored = []
            dark_pixels = 0

            for dy in range(-4, 5):
                for dx in range(-4, 5):
                    y, x = cy + dy, cx + dx
                    if 0 <= y < rgb.shape[0] and 0 <= x < rgb.shape[1]:
                        rr, gg, bb = rgb[y, x]
                        h, s, v = colorsys.rgb_to_hsv(rr / 255, gg / 255, bb / 255)

                        if s > 0.28 and v > 0.25:
                            colored.append((h, s, v, rr, gg, bb))

                        if rr < 95 and gg < 95 and bb < 95:
                            dark_pixels += 1

            if not colored:
                continue

            hues = [p[0] for p in colored]
            avg_h = sum(p[0] for p in colored) / len(colored)
            avg_s = sum(p[1] for p in colored) / len(colored)
            avg_v = sum(p[2] for p in colored) / len(colored)
            npix = len(colored)
            hue_span = max(hues) - min(hues)
            dark_ratio = dark_pixels / 81.0

            if 0.25 <= avg_h <= 0.45:
                hazards[(r, c)] = TELEPORT
                continue

            if 0.72 <= avg_h <= 0.90:
                hazards[(r, c)] = CONFUSION
                continue

            if dark_ratio > 0.12 and npix >= 18:
                hazards[(r, c)] = CONFUSION
                continue

            if npix >= 26 and hue_span < 0.12 and avg_v > 0.65 and avg_s > 0.40:
                hazards[(r, c)] = TELEPORT
            else:
                hazards[(r, c)] = DEATH_PIT

    return hazards


# ================================================================
# 3. Merge hazards into the matrix
# ================================================================
def merge_hazards_into_matrix(base_matrix, hazards_dict):
    merged = base_matrix.copy()

    for (r, c), hazard_type in hazards_dict.items():
        merged[r * 2, c * 2] = hazard_type

    return merged


def build_hazard_maze_matrix(base_maze_path, hazard_maze_path):
    base_matrix = load_maze(base_maze_path)
    hazards_dict = detect_hazards(hazard_maze_path)
    merged_matrix = merge_hazards_into_matrix(base_matrix, hazards_dict)
    return merged_matrix, hazards_dict


# ================================================================
# 4. Find start and goal
# ================================================================
def find_start_and_goal(image_path):
    gray = np.array(Image.open(image_path).convert("L"))

    top = [c for c in range(gray.shape[1]) if gray[1, c] > 128]
    bottom = [c for c in range(gray.shape[1]) if gray[-2, c] > 128]

    tc = ((top[len(top) // 2] - BORDER) // STRIDE) * 2
    bc = ((bottom[len(bottom) // 2] - BORDER) // STRIDE) * 2

    return (0, tc), (NUM_CELLS * 2 - 2, bc)


# ================================================================
# 5. BFS solver (kept as baseline/debug helper)
# ================================================================
def solve(matrix, start, goal, blocked=None):
    if blocked is None:
        blocked = {WALL}

    queue = deque([start])
    came_from = {start: None}

    while queue:
        cur = queue.popleft()

        if cur == goal:
            path = []
            node = cur
            while node is not None:
                path.append(node)
                node = came_from[node]
            return path[::-1]

        r, c = cur
        for nr, nc in [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]:
            if (
                0 <= nr < MAT_SIZE
                and 0 <= nc < MAT_SIZE
                and (nr, nc) not in came_from
                and matrix[nr, nc] not in blocked
            ):
                came_from[(nr, nc)] = cur
                queue.append((nr, nc))

    return None


# ================================================================
# 6. Draw matrix / route images
# ================================================================
def save_matrix_image(matrix, out_path, scale=8, solution=None):
    size = MAT_SIZE * scale
    img = Image.new("RGB", (size, size))
    draw = ImageDraw.Draw(img)

    for r in range(MAT_SIZE):
        for c in range(MAT_SIZE):
            val = int(matrix[r, c])
            color = CELL_COLORS.get(val, (255, 0, 255))
            draw.rectangle(
                [c * scale, r * scale, c * scale + scale - 1, r * scale + scale - 1],
                fill=color
            )

    if solution:
        dot = max(2, scale // 3)
        for r, c in solution:
            cx = c * scale + scale // 2
            cy = r * scale + scale // 2
            draw.ellipse([cx - dot, cy - dot, cx + dot, cy + dot], fill=(255, 255, 0))

    img.save(out_path)


def save_matrix_image_with_labels(matrix, out_path, scale=8):
    start, goal = find_start_and_goal("MAZE_0.png")
    matrix_copy = matrix.copy()
    matrix_copy[start[0], start[1]] = START
    matrix_copy[goal[0], goal[1]] = GOAL
    save_matrix_image(matrix_copy, out_path, scale=scale)


def save_solution_image_on_original(image_path, solution, out_path):
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)

    def to_px(r, c):
        return (
            BORDER + (c / 2) * STRIDE + CELL_SIZE / 2,
            BORDER + (r / 2) * STRIDE + CELL_SIZE / 2
        )

    draw.line([to_px(r, c) for r, c in solution], fill=(220, 50, 50), width=3)

    for pos, color in [(solution[0], (0, 210, 0)), (solution[-1], (30, 100, 220))]:
        x, y = to_px(*pos)
        draw.ellipse([x - 7, y - 7, x + 7, y + 7], fill=color)

    img.save(out_path)


# ================================================================
# 7. Build teleport map
# ================================================================
def build_teleport_map(matrix):
    pads = [
        (r, c)
        for r in range(0, MAT_SIZE, 2)
        for c in range(0, MAT_SIZE, 2)
        if matrix[r, c] == TELEPORT
    ]

    mapping = {}
    for i in range(0, len(pads), 2):
        if i + 1 < len(pads):
            mapping[pads[i]] = pads[i + 1]
            mapping[pads[i + 1]] = pads[i]

    return mapping


# ================================================================
# 8. Environment
# ================================================================
class MazeEnvironment:
    def __init__(self, maze_id: str):
        if maze_id == "training":
            base_image = "MAZE_0.png"
            hazard_image = "MAZE_1.png"
        else:
            raise ValueError(f"Unknown maze_id: {maze_id}")

        self.base_image_path = base_image
        self.hazard_image_path = hazard_image

        self.matrix, self.hazards_dict = build_hazard_maze_matrix(base_image, hazard_image)

        self.start_mat, self.goal_mat = find_start_and_goal(base_image)
        self.teleport_map = build_teleport_map(self.matrix)

        self.start_xy = (self.start_mat[1] // 2, self.start_mat[0] // 2)
        self.goal_xy = (self.goal_mat[1] // 2, self.goal_mat[0] // 2)

        self.reset()

    def reset(self):
        self.pos = self.start_xy
        self.confused_left = 0
        self.turns = 0
        self.deaths = 0
        self.confused_hits = 0
        self.explored = set()
        self.goal_reached = False
        self.pending_respawn = False
        return self.pos

    def _cell_type(self, x, y):
        return int(self.matrix[y * 2, x * 2])

    def _try_move(self, action: Action, confused: bool):
        dx, dy = {
            Action.MOVE_UP: (0, -1),
            Action.MOVE_DOWN: (0, 1),
            Action.MOVE_LEFT: (-1, 0),
            Action.MOVE_RIGHT: (1, 0),
            Action.WAIT: (0, 0),
        }[action]

        if confused and action != Action.WAIT:
            dx, dy = -dx, -dy

        x, y = self.pos
        nx, ny = x + dx, y + dy

        if not (0 <= nx < NUM_CELLS and 0 <= ny < NUM_CELLS):
            return x, y, True

        if self.matrix[y * 2 + dy, x * 2 + dx] == WALL:
            return x, y, True

        return nx, ny, False

    def step(self, actions: List[Action]) -> TurnResult:
        if not actions or len(actions) > 5:
            raise ValueError("Need 1-5 actions per turn.")

        if self.pending_respawn:
            self.pos = self.start_xy
            self.pending_respawn = False

        res = TurnResult()

        turn_confused = self.confused_left > 0
        if turn_confused:
            self.confused_left -= 1
            res.is_confused = True

        got_confused = False

        for action in actions:
            nx, ny, wall_hit = self._try_move(action, turn_confused or got_confused)

            if wall_hit:
                res.wall_hits += 1
                res.actions_executed += 1
                continue

            self.pos = (nx, ny)
            self.explored.add(self.pos)
            res.actions_executed += 1

            cell = self._cell_type(nx, ny)

            if cell == DEATH_PIT:
                res.is_dead = True
                res.current_position = (nx, ny)
                self.deaths += 1
                self.pending_respawn = True
                break

            if cell == TELEPORT:
                key = (ny * 2, nx * 2)
                if key in self.teleport_map:
                    dst = self.teleport_map[key]
                    self.pos = (dst[1] // 2, dst[0] // 2)
                    res.teleported = True

            elif cell == CONFUSION:
                if not got_confused:
                    got_confused = True
                    self.confused_left = 1
                    res.is_confused = True
                    self.confused_hits += 1

            if self.pos == self.goal_xy:
                res.is_goal_reached = True
                self.goal_reached = True
                break

        if not res.is_dead:
            res.current_position = self.pos

        self.turns += 1
        return res

    def get_episode_stats(self):
        return {
            "turns_taken": self.turns,
            "deaths": self.deaths,
            "confused": self.confused_hits,
            "cells_explored": len(self.explored),
            "goal_reached": self.goal_reached,
        }


# ================================================================
# 9. Simple GeeksforGeeks-style Q-learning
# ================================================================
ACTION_LIST = [
    Action.MOVE_UP,
    Action.MOVE_DOWN,
    Action.MOVE_LEFT,
    Action.MOVE_RIGHT,
    Action.WAIT,
]

n_states = NUM_CELLS * NUM_CELLS * 2
n_actions = len(ACTION_LIST)

Q_table = np.zeros((n_states, n_actions))

learning_rate = 0.8
discount_factor = 0.95
exploration_prob = 0.4
epochs = 3000


def state_to_index(pos, confused):
    x, y = pos
    confused_bit = 1 if confused else 0
    return (y * NUM_CELLS + x) * 2 + confused_bit


def manhattan_distance(pos1, pos2):
    x1, y1 = pos1
    x2, y2 = pos2
    return abs(x1 - x2) + abs(y1 - y2)


def compute_reward(result, old_pos, new_pos, visited_cells, goal_pos):
    reward = -1

    if result.wall_hits > 0:
        reward -= 5

    if result.is_dead:
        reward -= 100

    if result.is_goal_reached:
        reward += 500

    if new_pos not in visited_cells:
        reward += 5

    old_dist = manhattan_distance(old_pos, goal_pos)
    new_dist = manhattan_distance(new_pos, goal_pos)

    if new_dist < old_dist:
        reward += 2
    elif new_dist > old_dist:
        reward -= 2

    if old_pos == new_pos:
        reward -= 2

    return reward


def train_q_learning(env):
    global exploration_prob

    for epoch in range(epochs):
        current_pos = env.reset()
        visited_cells = set()
        visited_cells.add(current_pos)

        while True:
            current_state = state_to_index(current_pos, env.confused_left > 0)

            if np.random.rand() < exploration_prob:
                action_index = np.random.randint(0, n_actions)
            else:
                action_index = int(np.argmax(Q_table[current_state]))

            action = ACTION_LIST[action_index]

            old_pos = current_pos
            result = env.step([action])
            current_pos = result.current_position

            next_state = state_to_index(current_pos, result.is_confused)
            reward = compute_reward(result, old_pos, current_pos, visited_cells, env.goal_xy)

            Q_table[current_state, action_index] += learning_rate * (
                reward
                + discount_factor * np.max(Q_table[next_state])
                - Q_table[current_state, action_index]
            )

            visited_cells.add(current_pos)

            if result.is_goal_reached:
                break

            if env.turns >= 10000:
                break

        exploration_prob = max(0.10, exploration_prob * 0.999)

        if (epoch + 1) % 200 == 0:
            stats = env.get_episode_stats()
            print(
                f"Episode {epoch + 1}: "
                f"goal={stats['goal_reached']} "
                f"turns={stats['turns_taken']} "
                f"deaths={stats['deaths']} "
                f"cells_explored={stats['cells_explored']}"
            )


def evaluate_q_learning(env, test_episodes=5):
    successes = 0
    total_turns = 0
    total_deaths = 0
    successful_path_lengths = []

    for _ in range(test_episodes):
        current_pos = env.reset()
        path_length = 0

        while True:
            current_state = state_to_index(current_pos, env.confused_left > 0)
            action_index = int(np.argmax(Q_table[current_state]))
            action = ACTION_LIST[action_index]

            result = env.step([action])
            current_pos = result.current_position
            path_length += 1

            if result.is_goal_reached:
                successes += 1
                successful_path_lengths.append(path_length)
                break

            if env.turns >= 10000:
                break

        total_turns += env.turns
        total_deaths += env.deaths

    success_rate = successes / test_episodes
    avg_turns = total_turns / test_episodes
    avg_path_length = (
        sum(successful_path_lengths) / len(successful_path_lengths)
        if successful_path_lengths else 0
    )
    death_rate = total_deaths / total_turns if total_turns > 0 else 0

    print("\nFinal metrics:")
    print("success_rate =", success_rate)
    print("avg_turns =", avg_turns)
    print("avg_path_length =", avg_path_length)
    print("death_rate =", death_rate)

    return {
        "success_rate": success_rate,
        "avg_turns": avg_turns,
        "avg_path_length": avg_path_length,
        "death_rate": death_rate,
    }


def run_greedy_episode_and_save_path(env, max_turns=10000):
    current_pos = env.reset()
    route = [(current_pos[1] * 2, current_pos[0] * 2)]
    visited_in_episode = set()
    visited_in_episode.add(current_pos)

    while env.turns < max_turns:
        current_state = state_to_index(current_pos, env.confused_left > 0)
        action_index = int(np.argmax(Q_table[current_state]))
        action = ACTION_LIST[action_index]

        result = env.step([action])
        current_pos = result.current_position
        route.append((current_pos[1] * 2, current_pos[0] * 2))

        if result.is_goal_reached:
            return route, True

        if current_pos in visited_in_episode and not result.is_dead:
            if len(route) > 2000:
                break

        visited_in_episode.add(current_pos)

    return route, False


# ================================================================
# 10. Main
# ================================================================
if __name__ == "__main__":
    # Save base matrix image
    matrix0 = load_maze("MAZE_0.png")
    save_matrix_image_with_labels(matrix0, "MAZE_0_matrix.png", scale=8)

    # Save hazard matrix image
    matrix1, hazards_dict = build_hazard_maze_matrix("MAZE_0.png", "MAZE_1.png")
    save_matrix_image_with_labels(matrix1, "MAZE_1_matrix_with_hazards.png", scale=8)

    print("Hazard counts:")
    death_pits = sum(1 for h in hazards_dict.values() if h == DEATH_PIT)
    teleports = sum(1 for h in hazards_dict.values() if h == TELEPORT)
    confusions = sum(1 for h in hazards_dict.values() if h == CONFUSION)
    print("Death pits:", death_pits)
    print("Teleports :", teleports)
    print("Confusion :", confusions)

    env = MazeEnvironment("training")

    print("\nTraining Q-learning agent...")
    train_q_learning(env)

    print("\nEvaluating learned policy...")
    metrics = evaluate_q_learning(env, test_episodes=5)

    route, solved = run_greedy_episode_and_save_path(env)
    print("Solved greedy route after training:", solved)
    print("Route length recorded:", len(route))

    if solved:
        save_matrix_image(env.matrix, "RL_learned_route_matrix.png", scale=8, solution=route)
        save_solution_image_on_original("MAZE_1.png", route, "RL_learned_route_original.png")
        print("Saved RL learned route images.")
    else:
        print("RL policy did not reach the goal, so no solved route image was produced.")

    print("\nSaved image files:")
    print(" - MAZE_0_matrix.png")
    print(" - MAZE_1_matrix_with_hazards.png")
    if solved:
        print(" - RL_learned_route_matrix.png")
        print(" - RL_learned_route_original.png")

    print("\nLearned Q-table shape:", Q_table.shape)

/tmp/ipykernel_37620/4174411142.py:88: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  gray = np.array(Image.open(image_path).convert("L"))
/tmp/ipykernel_37620/4174411142.py:189: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  gray = np.array(Image.open(image_path).convert("L"))
/tmp/ipykernel_37620/4174411142.py:111: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arg

Hazard counts:
Death pits: 33
Teleports : 8
Confusion : 2

Training Q-learning agent...
Episode 200: goal=False turns=10000 deaths=0 cells_explored=88
Episode 400: goal=False turns=10000 deaths=0 cells_explored=76
Episode 600: goal=False turns=10000 deaths=0 cells_explored=78
Episode 800: goal=False turns=10000 deaths=0 cells_explored=92
Episode 1000: goal=False turns=10000 deaths=0 cells_explored=49
Episode 1200: goal=False turns=10000 deaths=0 cells_explored=52
Episode 1400: goal=False turns=10000 deaths=0 cells_explored=48
Episode 1600: goal=False turns=10000 deaths=0 cells_explored=75
Episode 1800: goal=False turns=10000 deaths=0 cells_explored=81
Episode 2000: goal=False turns=10000 deaths=0 cells_explored=75
Episode 2200: goal=False turns=10000 deaths=0 cells_explored=75
Episode 2400: goal=False turns=10000 deaths=0 cells_explored=75
Episode 2600: goal=False turns=10000 deaths=0 cells_explored=75
Episode 2800: goal=False turns=10000 deaths=0 cells_explored=52
Episode 3000: goal=F